In [ ]:
# Import necessary libraries
import os
import sys
from dotenv import load_dotenv
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text

sys.path.insert(0, '..')
from src.transformation import assign_customer_ids

load_dotenv()

In [ ]:
# Ingest the Dreamers Ecommerce dataset
dreamers_df = pd.read_csv('../dataset/raw_data/dreamers_ecommerce.csv')

In [ ]:
# Preserve genuine IDs; create surrogates only for anonymous invoices
dreamers_df["CustomerID"] = assign_customer_ids(dreamers_df)

In [ ]:
# Create Invoice DataFrame
invoices = (
    dreamers_df[["InvoiceNo", "CustomerID", "InvoiceDate"]]
    .drop_duplicates(subset=["InvoiceNo"])
    .reset_index(drop=True)
)

In [ ]:
# Create Customer DataFrame
customers = (
    dreamers_df[["CustomerID", "Country"]]
    .drop_duplicates(subset=["CustomerID"])
    .reset_index(drop=True)
)

In [ ]:
# Create Product DataFrame
products = (
    dreamers_df[["StockCode", "Description"]]
    .drop_duplicates(subset=["StockCode"])
    .reset_index(drop=True)
)

In [ ]:
# Create Invoice Items DataFrame
invoice_items = (
    dreamers_df.groupby(["InvoiceNo", "StockCode", "UnitPrice"], as_index=False)["Quantity"].sum()
)

In [ ]:
# Save processed DataFrames to CSV files
invoices.to_csv('../dataset/processed_data/invoice.csv', index=False)
customers.to_csv('../dataset/processed_data/customer.csv', index=False)
products.to_csv('../dataset/processed_data/product.csv', index=False)
invoice_items.to_csv('../dataset/processed_data/invoice_items.csv', index=False)

In [ ]:
# Create a database in PostgreSQL and store the data in a table
# Define the database connection parameters
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

In [ ]:
# Create the PostgreSQL database if it does not already exist
admin_engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/postgres",
    isolation_level="AUTOCOMMIT",
)
with admin_engine.connect() as connection:
    database_exists = connection.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :db_name"),
        {"db_name": db_name},
    ).scalar()
    if not database_exists:
        quoted_db_name = admin_engine.dialect.identifier_preparer.quote_identifier(
            db_name
        )
        connection.exec_driver_sql(f"CREATE DATABASE {quoted_db_name}")

admin_engine.dispose()
engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

In [ ]:
# Build the new tables in a staging schema.
with engine.begin() as connection:
    connection.execute(text("DROP SCHEMA IF EXISTS dreamers_staging CASCADE"))
    connection.execute(text("CREATE SCHEMA dreamers_staging"))

    # Create constrained destination tables.
    connection.execute(text("""
        CREATE TABLE dreamers_staging.customers (
            "CustomerID" INT PRIMARY KEY,
            "Country" VARCHAR(255)
        );

        CREATE TABLE dreamers_staging.products (
            "StockCode" VARCHAR(255) PRIMARY KEY,
            "Description" VARCHAR(255)
        );

        CREATE TABLE dreamers_staging.invoices (
            "InvoiceNo" VARCHAR(255) PRIMARY KEY,
            "CustomerID" INT REFERENCES dreamers_staging.customers("CustomerID"),
            "InvoiceDate" TIMESTAMP
        );

        CREATE TABLE dreamers_staging.invoice_items (
            "InvoiceNo" VARCHAR(255) REFERENCES dreamers_staging.invoices("InvoiceNo"),
            "StockCode" VARCHAR(255) REFERENCES dreamers_staging.products("StockCode"),
            "UnitPrice" DECIMAL(10, 2),
            "Quantity" INT,
            PRIMARY KEY ("InvoiceNo", "StockCode", "UnitPrice")
        );
    """))

In [ ]:
# Load the saved cleaned data into the PostgreSQL database
with engine.begin() as connection:
    customers.to_sql(
        "customers",
        connection,
        schema="dreamers_staging",
        if_exists="append",
        index=False,
    )
    products.to_sql(
            "products",
            connection,
            schema="dreamers_staging",
            if_exists="append",
            index=False,
        )
    invoices.to_sql(
            "invoices",
            connection,
            schema="dreamers_staging",
            if_exists="append",
            index=False,
        )
    invoice_items.to_sql(
            "invoice_items",
            connection,
            schema="dreamers_staging",
            if_exists="append",
            index=False,
        )
    live_schema_exists = connection.execute(text(
        "SELECT 1 FROM information_schema.schemata WHERE schema_name = 'dreamers'"
    )).scalar()
    connection.execute(text("DROP SCHEMA IF EXISTS dreamers_previous CASCADE"))
    if live_schema_exists:
        connection.execute(text("ALTER SCHEMA dreamers RENAME TO dreamers_previous"))
    connection.execute(text("ALTER SCHEMA dreamers_staging RENAME TO dreamers"))
print("Data loaded into PostgreSQL database successfully.")

#### SQL QUERIES    

In [ ]:
sql_task_1 = """
-- 1. Total number of customers
SELECT COUNT(*) AS total_customers
FROM dreamers.customers;
"""
result = pd.read_sql(sql_task_1, engine)
result

In [ ]:
sql_task_2 = """
-- 2. Total number of products
SELECT COUNT(*) AS total_products
FROM dreamers.products;
"""
result = pd.read_sql(sql_task_2, engine)
result

In [ ]:
sql_task_9 = """
-- 9. Ten products with the highest quantity sold
SELECT p."StockCode", p."Description", SUM(i."Quantity") AS total_quantity
FROM dreamers.invoice_items AS i
JOIN dreamers.products AS p
    ON i."StockCode" = p."StockCode"
GROUP BY p."StockCode", p."Description"
ORDER BY total_quantity DESC
LIMIT 10;
"""
result = pd.read_sql(sql_task_9, engine)
result

In [ ]:
sql_task_10 = """
-- 10. Ten invoices with the highest sales value
SELECT i."InvoiceNo",
       ROUND(SUM(i."Quantity" * i."UnitPrice"), 2) AS total_sales
FROM dreamers.invoice_items AS i
GROUP BY i."InvoiceNo"
ORDER BY total_sales DESC
LIMIT 10;
"""
result = pd.read_sql(sql_task_10, engine)
result